In [5]:
import torch
import pandas as pd
from torch.utils.data import DataLoader,Dataset,random_split
import torch.optim as optim
import torch.nn as nn
import os
import copy

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prasoonkottarathil/polycystic-ovary-syndrome-pcos")

print("Path to dataset files:", path)

100%|██████████| 120k/120k [00:00<00:00, 63.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/prasoonkottarathil/polycystic-ovary-syndrome-pcos/versions/3


In [3]:
import os

dataset_path = "/root/.cache/kagglehub/datasets/prasoonkottarathil/polycystic-ovary-syndrome-pcos/versions/3"

print(os.listdir(dataset_path))

['PCOS_infertility.csv', 'PCOS_data_without_infertility.xlsx']


In [9]:
import pandas as pd

file_path = "/root/.cache/kagglehub/datasets/prasoonkottarathil/polycystic-ovary-syndrome-pcos/versions/3/PCOS_data_without_infertility.xlsx"

# inspect sheet names first
xls = pd.ExcelFile(file_path)
print(xls.sheet_names)

['Instructions', 'Full_new']


In [15]:
df = pd.read_excel(file_path, sheet_name="Full_new")

df.head()
df.columns

Index(['Sl. No', 'Patient File No.', 'PCOS (Y/N)', ' Age (yrs)', 'Weight (Kg)',
       'Height(Cm) ', 'BMI', 'Blood Group', 'Pulse rate(bpm) ',
       'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)',
       'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions',
       '  I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)',
       'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio',
       'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)',
       'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
       'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)',
       'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)',
       'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)',
       'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)',
       'Unnamed: 44'],
      dtype='object')

In [18]:
# remove useless column
df = df.drop(columns=['Sl. No', 'Patient File No.'])
# clean column names
df.columns = df.columns.str.strip()

print(df.columns)

Index(['PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI',
       'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)',
       'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)',
       'Pregnant(Y/N)', 'No. of aborptions', 'I   beta-HCG(mIU/mL)',
       'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH',
       'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)',
       'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)',
       'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
       'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)',
       'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)',
       'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)',
       'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)'],
      dtype='object')


In [22]:
missing = df.isnull().sum()
missing = missing[missing > 0]

print(missing.sort_values(ascending=False))

Marraige Status (Yrs)    1
Fast food (Y/N)          1
dtype: int64


In [23]:
# fill numerical column with median
df['Marraige Status (Yrs)'] = df['Marraige Status (Yrs)'].fillna(
    df['Marraige Status (Yrs)'].median()
)

# fill binary categorical with mode
df['Fast food (Y/N)'] = df['Fast food (Y/N)'].fillna(
    df['Fast food (Y/N)'].mode()[0]
)

In [24]:
print(df.isnull().sum().sum())

0


In [20]:
target = 'PCOS (Y/N)'

In [21]:
print(df[target].value_counts())

PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64


In [25]:
print(df.duplicated().sum())

0


In [26]:
len(df.columns)

42

In [28]:
numeric_cols = [
    'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI',
    'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)',
    'Cycle length(days)', 'Marraige Status (Yrs)',
    'No. of aborptions', 'FSH(mIU/mL)', 'LH(mIU/mL)',
    'FSH/LH', 'Hip(inch)', 'Waist(inch)',
    'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)',
    'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)',
    'RBS(mg/dl)', 'BP _Systolic (mmHg)',
    'BP _Diastolic (mmHg)', 'Follicle No. (L)',
    'Follicle No. (R)', 'Avg. F size (L) (mm)',
    'Avg. F size (R) (mm)', 'Endometrium (mm)'
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'[^0-9.]', '', regex=True)
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 42 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   PCOS (Y/N)              541 non-null    int64  
 1   Age (yrs)               541 non-null    int64  
 2   Weight (Kg)             541 non-null    float64
 3   Height(Cm)              541 non-null    float64
 4   BMI                     541 non-null    float64
 5   Blood Group             541 non-null    int64  
 6   Pulse rate(bpm)         541 non-null    int64  
 7   RR (breaths/min)        541 non-null    int64  
 8   Hb(g/dl)                541 non-null    float64
 9   Cycle(R/I)              541 non-null    int64  
 10  Cycle length(days)      541 non-null    int64  
 11  Marraige Status (Yrs)   541 non-null    float64
 12  Pregnant(Y/N)           541 non-null    int64  
 13  No. of aborptions       541 non-null    int64  
 14  I   beta-HCG(mIU/mL)    541 non-null    fl

In [31]:
df['II    beta-HCG(mIU/mL)'] = (
    df['II    beta-HCG(mIU/mL)']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)
)

df['II    beta-HCG(mIU/mL)'] = pd.to_numeric(
    df['II    beta-HCG(mIU/mL)'],
    errors='coerce'
)

In [32]:
df['II    beta-HCG(mIU/mL)'] = df['II    beta-HCG(mIU/mL)'].fillna(
    df['II    beta-HCG(mIU/mL)'].median()
)
df['AMH(ng/mL)'] = df['AMH(ng/mL)'].fillna(
    df['AMH(ng/mL)'].median()
)

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['PCOS (Y/N)'])
y = df['PCOS (Y/N)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

                   Feature  Importance
37        Follicle No. (R)    0.200697
36        Follicle No. (L)    0.129440
29    Skin darkening (Y/N)    0.062881
27        Weight gain(Y/N)    0.050934
28        hair growth(Y/N)    0.044951
22              AMH(ng/mL)    0.042900
8               Cycle(R/I)    0.029220
16              LH(mIU/mL)    0.023599
17                  FSH/LH    0.022474
9       Cycle length(days)    0.021378
0                Age (yrs)    0.020833
15             FSH(mIU/mL)    0.018273
21             TSH (mIU/L)    0.016822
1              Weight (Kg)    0.016746
18               Hip(inch)    0.016394
10   Marraige Status (Yrs)    0.016293
40        Endometrium (mm)    0.016142
3                      BMI    0.015695
39    Avg. F size (R) (mm)    0.015560
31            Pimples(Y/N)    0.015271
38    Avg. F size (L) (mm)    0.015117
20         Waist:Hip Ratio    0.015068
25              PRG(ng/mL)    0.014744
24          Vit D3 (ng/mL)    0.014609
26              RBS(mg/dl

In [34]:
top_features = [
    'Follicle No. (R)',
    'Follicle No. (L)',
    'Skin darkening (Y/N)',
    'Weight gain(Y/N)',
    'hair growth(Y/N)',
    'AMH(ng/mL)',
    'Cycle(R/I)',
    'LH(mIU/mL)',
    'FSH/LH',
    'Cycle length(days)'
]

In [38]:
X = df[top_features]
y = df['PCOS (Y/N)']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [39]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[69  4]
 [ 6 30]]
              precision    recall  f1-score   support

           0       0.92      0.95      0.93        73
           1       0.88      0.83      0.86        36

    accuracy                           0.91       109
   macro avg       0.90      0.89      0.89       109
weighted avg       0.91      0.91      0.91       109



In [40]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[71  2]
 [ 7 29]]
              precision    recall  f1-score   support

           0       0.91      0.97      0.94        73
           1       0.94      0.81      0.87        36

    accuracy                           0.92       109
   macro avg       0.92      0.89      0.90       109
weighted avg       0.92      0.92      0.92       109



In [51]:

from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
0.8168723217224116


In [46]:
param_dist = {
    'n_estimators': [50, 100, 150, 200],

    'max_depth': [2, 3, 4, 5, 6],

    'learning_rate': [0.01, 0.03, 0.05, 0.1],

    'subsample': [0.7, 0.8, 0.9, 1.0],

    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],

    'min_child_weight': [1, 3, 5],

    'gamma': [0, 0.1, 0.2, 0.3]
}
from sklearn.model_selection import RandomizedSearchCV
random = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
random.fit(X_train,y_train)
print(random.best_params_)
print(random.best_score_)

{'subsample': 0.8, 'n_estimators': 150, 'min_child_weight': 3, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.8}
0.8339063231850117


In [48]:
random_model = random.best_estimator_

In [52]:
grid_model = grid.best_estimator_

In [54]:
y_pred = random_model.predict(X_test)
print(classification_report(y_test, y_pred))
y_pred = grid_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.97      0.94        73
           1       0.94      0.81      0.87        36

    accuracy                           0.92       109
   macro avg       0.92      0.89      0.90       109
weighted avg       0.92      0.92      0.92       109

              precision    recall  f1-score   support

           0       0.92      0.97      0.95        73
           1       0.94      0.83      0.88        36

    accuracy                           0.93       109
   macro avg       0.93      0.90      0.91       109
weighted avg       0.93      0.93      0.93       109



In [55]:
best_model = grid_model

In [56]:
from sklearn.model_selection import cross_val_score
import numpy as np

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Score:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

Cross Validation Scores: [0.90825688 0.88888889 0.83333333 0.88888889 0.80555556]
Mean CV Score: 0.8649847094801224
Std Dev: 0.03885597596151335


In [57]:
import joblib

joblib.dump(best_model, "pcos_grid_model.pkl")

['pcos_grid_model.pkl']

In [58]:
from google.colab import files

files.download("pcos_grid_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
print(X.columns.tolist())

['Follicle No. (R)', 'Follicle No. (L)', 'Skin darkening (Y/N)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'AMH(ng/mL)', 'Cycle(R/I)', 'LH(mIU/mL)', 'FSH/LH', 'Cycle length(days)']
